# Notebook 09 layered FMM-inspired energy encoder

Definitions only: no data loading, training, optimizer steps or long run. Inputs are images plus visible original patch indices and validity; outputs match the existing JEPA encoder interface. Run top-to-bottom in a fresh `ahr-go` kernel (under ten seconds) or import only tagged `ahr-export` definitions. This notebook has no checkpoints; restart the kernel and Run All to reload. The main Notebook 09 owns any future experiment checkpoints and exact resume instructions.

Four variants share a four-block, dimension-192 patch/MLP scaffold: current ViT; MLP without patch mixing; local energy corrections; local plus hierarchical far energy corrections. For an energy block, u = LayerNorm(x); initialize z0 = u; take one default step z1 = u - 0.5 grad E(u | u); then x <- x + W(z1-u), followed by x <- x + MLP(LayerNorm(x)). W has no bias and the MLP expands by 4. Every block owns independently learned coefficients. Configured two-step refinement is supported without changing model parameters.

The scalar energy and adjoint operations reuse Notebook 07 tagged definitions. Coefficients are fixed within each block's inner refinement but recomputed from the next block's input. Degree-normalized factors, diagonal contractions, and the exact spectral projection bound give each inner Hessian in [1,2] at strengths 0.25/0.25; step 0.5 is monotone and contracts inner distance by at least 1/2. The surrounding learned projection, residuals, MLPs and changing energies have no global contraction guarantee.

The MLP control has no unused energy parameters; its retained norm1 is unused and counted separately. The local control has no far coefficients or summary projection. Encoder sizes therefore differ and are reported explicitly rather than described as parameter matched. Context gathering is mask safe before learned embedding; both context and EMA teacher use the selected encoder; the existing Transformer predictor and JEPA loss stay unchanged. Energy encoder computations use FP32 (FP64 for audits); there is no attention or sampling hidden in the energy mixer.

In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_namespace = {}
for _cell in json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _namespace)
_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Notebook09: independently learned energy corrections inside deep MLP blocks."""
import copy
import hashlib
import json
import math
from pathlib import Path
import sys
import types
import torch
from torch import nn
from torch.nn import functional as F
from fmm_jepa.models import JEPA, Predictor

BACKEND = "deep-fmm-energy-v1"
VARIANTS = ("vit", "mlp", "local", "local_far")


def _load_energy07_definitions():
    """Import only immutable Notebook07 ahr-export definitions, never its runners."""
    root=Path(sys.modules["fmm_jepa"].__project_root__)
    path=root/"experiments/07_energy/encoder.ipynb"
    name="fmm09_energy07_definitions"
    if name not in sys.modules:
        cells=json.loads(path.read_text())["cells"]
        sources=["".join(c["source"]) for c in cells if c["cell_type"]=="code" and "ahr-export" in c.get("metadata",{}).get("tags",[])]
        if not sources: raise ImportError(f"No tagged exports in {path}")
        module=types.ModuleType(name);module.__file__=str(path)
        module.__export_sha256__=hashlib.sha256("\n".join(sources).encode()).hexdigest()
        sys.modules[name]=module
        try:
            for i,source in enumerate(sources): exec(compile(source,str(path)+f"#ahr-export-{i}","exec"),module.__dict__)
        except Exception:
            del sys.modules[name]
            raise
    return sys.modules[name]


ENERGY07=_load_energy07_definitions()


class ConditionalEnergyStep(ENERGY07.EnergyEncoder):
    """Local or local+far convex energy; no patch stem or unused far parameters."""
    def __init__(self,dim=192,summary_dim=32,leaf_size=2,inner_steps=1,near_strength=.25,
                 far_strength=.25,step_size=.5,coefficient_dim=32,checkpoint=True,
                 edge_tile=256,variant="local_far"):
        nn.Module.__init__(self)
        if variant not in ("local","local_far"): raise ValueError("Energy step requires local or local_far")
        if min(dim,summary_dim,leaf_size,inner_steps,coefficient_dim,edge_tile)<1:
            raise ValueError("Positive dimensions, tile width and inner_steps required")
        effective_far=float(far_strength) if variant=="local_far" else 0.
        if not math.isfinite(near_strength) or not math.isfinite(effective_far) or min(near_strength,effective_far)<0:
            raise ValueError("Finite nonnegative strengths required")
        bound=1+2*near_strength+2*effective_far
        if not 0<float(step_size)<=1/bound: raise ValueError("step_size exceeds certified monotone bound")
        self.dim,self.summary_dim,self.leaf_size=int(dim),int(summary_dim),int(leaf_size)
        self.refinement_steps,self.inner_steps=int(inner_steps),int(inner_steps)
        self.near_strength,self.far_strength=float(near_strength),effective_far
        self.step_size,self.checkpoint_refinement=float(step_size),bool(checkpoint)
        self.edge_tile,self.variant,self.interaction_variant=int(edge_tile),"energy",variant
        self.near_coeff=nn.Sequential(nn.LayerNorm(dim),nn.Linear(dim,coefficient_dim),nn.GELU(),nn.Linear(coefficient_dim,2*dim+1))
        self.near_position=nn.Linear(2,2*dim+1)
        initializers=[self.near_coeff[-1],self.near_position]
        if variant=="local_far":
            self.summary_raw=nn.Parameter(torch.randn(summary_dim,dim)/math.sqrt(dim))
            self.far_coeff=nn.Sequential(nn.LayerNorm(summary_dim),nn.Linear(summary_dim,coefficient_dim),nn.GELU(),nn.Linear(coefficient_dim,2*summary_dim+1))
            self.far_position=nn.Linear(2,2*summary_dim+1)
            initializers.extend((self.far_coeff[-1],self.far_position))
        for layer in initializers:
            nn.init.normal_(layer.weight,std=.02);nn.init.zeros_(layer.bias)

    def prepare_energy(self,h,indices,valid,grid):
        # A factor's coefficients depend on h once per block, never its iterates z.
        h=torch.where(valid[...,None],h,torch.zeros_like(h))
        b,v,_=h.shape
        geometry=ENERGY07.device_geometry(tuple(grid),self.leaf_size,h.device)
        full_count=int(grid[0])*int(grid[1])
        slots=torch.where(valid,indices,full_count)
        lookup=torch.full((b,full_count+1),v,device=h.device,dtype=torch.long)
        lookup=lookup.scatter(1,slots,torch.arange(v,device=h.device)[None,:].expand(b,-1))
        ni,nj=lookup[:,geometry["near_i"]],lookup[:,geometry["near_j"]]
        active=(ni<v)&(nj<v);ni,nj=ni.clamp_max(v-1),nj.clamp_max(v-1)
        state={"h":h,"valid":valid,"indices":indices,"geometry":geometry,
            "near_i":ni,"near_j":nj,"near_base":ENERGY07._degree_weights(ni,nj,active,v,h.dtype),
            "near_node":self.near_coeff(h),"near_position":self.near_position(geometry["near_xy"].to(h.dtype))}
        if self.interaction_variant=="local_far":
            state["leaf_index"]=geometry["cells"][indices.clamp(0,full_count-1)]
            state["projection"]=self.summary_projection()
            state["counts"]=self._tree_sums(valid[...,None].to(h.dtype),state)
            hsums=self._tree_sums(F.linear(h,state["projection"])*valid[...,None],state)
            state["far"]=[]
            interacting=sum(int(level["i"].numel()>0) for level in geometry["levels"])
            for level,count,hsum in zip(geometry["levels"],state["counts"],hsums):
                i,j=level["i"],level["j"]
                active=(ENERGY07._gather(count,i)[...,0]>0)&(ENERGY07._gather(count,j)[...,0]>0)
                state["far"].append({"node":self.far_coeff(hsum/count.clamp_min(1)),
                    "position":self.far_position(level["xy"].to(h.dtype)),
                    "base":ENERGY07._degree_weights(i,j,active,level["side"]**2,h.dtype),
                    "alpha":1/max(1,interacting) if i.numel() else 0.})
        return state

    def energy(self,z,state,return_terms=False):
        anchor=.5*((z-state["h"]).square()*state["valid"][...,None]).sum((1,2))
        near=self.near_strength*self._factors(z,state["near_node"],state["near_position"],state["near_i"],state["near_j"],state["near_base"],False)
        far=anchor*0
        if self.far_strength:
            sums=self._tree_sums(F.linear(z,state["projection"])*state["valid"][...,None],state)
            for level,count,s,factors in zip(state["geometry"]["levels"],state["counts"],sums,state["far"]):
                far=far+self.far_strength*factors["alpha"]*self._factors(s/count.clamp_min(1),factors["node"],factors["position"],level["i"],level["j"],factors["base"],False)
        terms={"anchor":anchor,"near":near,"far":far,"total":anchor+near+far}
        return terms if return_terms else terms["total"]

    def forward(self,h,indices,valid,grid,return_trace=False):
        state=self.prepare_energy(h,indices,valid,grid)
        return self.refine(state["h"],state,return_trace=return_trace)


class EnergyCorrection(nn.Module):
    """Project the displacement from the block input: W(z_K-u), without bias."""
    def __init__(self,dim=192,**options):
        super().__init__()
        self.energy=ConditionalEnergyStep(dim=dim,**options)
        self.projection=nn.Linear(dim,dim,bias=False)

    def forward(self,u,indices,valid,grid):
        u=torch.where(valid[...,None],u,torch.zeros_like(u))
        z=self.energy(u,indices,valid,grid)
        return torch.where(valid[...,None],self.projection(z-u),torch.zeros_like(u))


class DeepEnergyBlock(nn.Module):
    def __init__(self,variant,dim,**options):
        super().__init__()
        self.norm1,self.norm2=nn.LayerNorm(dim),nn.LayerNorm(dim)
        self.mlp=nn.Sequential(nn.Linear(dim,4*dim),nn.GELU(),nn.Linear(4*dim,dim))
        # A parameterless absent mixer is explicit; no dummy attention/energy.
        self.mixer=None if variant=="mlp" else EnergyCorrection(dim=dim,variant=variant,**options)

    def forward(self,x,indices,valid,grid):
        if self.mixer is not None:
            x=x+self.mixer(self.norm1(x),indices,valid,grid)
        x=x+self.mlp(self.norm2(x))
        return torch.where(valid[...,None],x,torch.zeros_like(x))


class DeepEnergyEncoder(nn.Module):
    """A mask-safe ViT-shaped residual/MLP scaffold with independent energy blocks."""
    def __init__(self,kind="local_far",patch_size=8,dim=192,depth=4,heads=4,
                 summary_dim=32,leaf_size=2,inner_steps=1,near_strength=.25,
                 far_strength=.25,step_size=.5,coefficient_dim=32,checkpoint=True,edge_tile=256):
        super().__init__()
        if kind not in ("mlp","local","local_far"): raise ValueError("Expected mlp/local/local_far")
        if min(patch_size,dim,depth)<1: raise ValueError("Positive patch_size, dimension and depth required")
        self.patch_size,self.dim,self.kind=int(patch_size),int(dim),kind
        self.patch_embed=nn.Linear(3*patch_size*patch_size,dim)
        self.blocks=nn.ModuleList([DeepEnergyBlock(kind,dim,summary_dim=summary_dim,leaf_size=leaf_size,
            inner_steps=inner_steps,near_strength=near_strength,far_strength=far_strength,step_size=step_size,
            coefficient_dim=coefficient_dim,checkpoint=checkpoint,edge_tile=edge_tile) for _ in range(depth)])
        self.norm=nn.LayerNorm(dim)

    def embed(self,images,indices=None,valid=None):
        # Reuse audited raw-visible gather before patch embedding and fixed positions.
        return ENERGY07.EnergyEncoder.embed(self,images,indices,valid)

    def forward(self,images,indices=None,valid=None):
        with torch.autocast(device_type=images.device.type,enabled=False):
            images=images.to(self.patch_embed.weight.dtype)
            x,indices,valid,grid=self.embed(images,indices,valid)
            for block in self.blocks: x=block(x,indices,valid,grid)
            return torch.where(valid[...,None],self.norm(x),torch.zeros_like(x))


class DeepEnergyJEPA(JEPA):
    def __init__(self,config,variant=None):
        nn.Module.__init__(self)
        self.config=copy.deepcopy(config)
        options=config.get("deep09",{})
        variant=variant or options.get("variant","local_far")
        self.context=DeepEnergyEncoder(kind=variant,patch_size=config.get("patch_size",8),dim=config.get("dim",192),
            depth=options.get("depth",config.get("depth",4)),heads=config.get("heads",4),
            summary_dim=options.get("summary_dim",32),leaf_size=options.get("leaf_size",config.get("leaf_size",2)),
            inner_steps=options.get("inner_steps",options.get("steps",1)),near_strength=options.get("near_strength",.25),
            far_strength=options.get("far_strength",.25),step_size=options.get("step_size",.5),
            coefficient_dim=options.get("coefficient_dim",32),checkpoint=options.get("checkpoint",True),edge_tile=options.get("edge_tile",256))
        self.target=copy.deepcopy(self.context).requires_grad_(False).eval()
        self.predictor=Predictor(self.context.dim,dim=config.get("predictor_dim",32),depth=config.get("predictor_depth",1),
            heads=config.get("predictor_heads",config.get("heads",4)))


def build_model(config,kind="semantic",variant=None):
    variant=variant or config.get("deep09",{}).get("variant","local_far")
    if kind=="vit" or variant=="vit":
        cfg=copy.deepcopy(config);cfg["kind"]="vit"
        return JEPA(cfg)
    if kind!="semantic": raise ValueError("kind must be vit or semantic")
    return DeepEnergyJEPA(config,variant=variant)


def parameter_counts(model):
    """Explicit architecture size; inactive MLP-only norm1 parameters are separate."""
    encoder=model.context
    inactive=sum(p.numel() for block in encoder.blocks if block.mixer is None for p in block.norm1.parameters())
    count=lambda module:sum(p.numel() for p in module.parameters())
    return {"variant":encoder.kind,"encoder_total":count(encoder),"encoder_inactive_scaffold":inactive,
            "encoder_active":count(encoder)-inactive,"predictor":count(model.predictor),
            "trainable_total":sum(p.numel() for p in model.parameters() if p.requires_grad),
            "trainable_active":sum(p.numel() for p in model.parameters() if p.requires_grad)-inactive,
            "with_ema_teacher_total":count(model)}
